# Shared S1/voice directions with SVD (Qwen 0.5B, layer 18)

This notebook uses the **same frozen base-model SAE** for both independently trained LoRA adapters.

1. Load the committed S1 adapter, layer-18 SAE, and paired datasets.
2. Upload the paired-voice adapter.
3. Collect the residual state immediately after `Final answer:` for 100 matched S1 pairs and 100 matched voice pairs.
4. Encode every state with the same SAE.
5. Compute SVD in residual space (primary) and decoder-norm-scaled SAE feature space (feature discovery).
6. Download raw activations, decompositions, diagnostics, and top feature loadings.

Use a Colab GPU runtime. No SAE is retrained.

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU"
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
REPO_URL = "https://github.com/vladflorinfilip/Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography.git"

!rm -rf Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography
!git clone --depth 1 "{REPO_URL}"
%cd Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography
!pip install -q peft "transformers<4.50" accelerate tqdm
!pip uninstall -y torchao >/dev/null 2>&1

In [ ]:
from pathlib import Path
from google.colab import files
from transformers import AutoTokenizer

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
S1_ADAPTER = Path("checkpoints/qwen05b-cot-sft-v2")
VOICE_ADAPTER = Path("checkpoints/qwen05b-cot-sft-voice-paired")
SAE_PATH = Path("sparse_autoencoders/artifacts/ethics_l18/sae.pt")
S1_RECORDS = Path("data/evaluation_data/qwen/ETHICS/qwen05b_v1.jsonl")
S1_FLIPS = Path("data/intervention_data/qwen/ETHICS/interventions/negated_minimal_cot.jsonl")
VOICE_PAIRS = Path("data/validation_data/synthetic_ethics_voice_paired_val.jsonl")
RESULTS = Path("sparse_autoencoders/artifacts/ethics_l18/shared_svd")
RESULTS.mkdir(parents=True, exist_ok=True)

for path in (S1_ADAPTER / "adapter_model.safetensors", SAE_PATH, S1_RECORDS, S1_FLIPS, VOICE_PAIRS):
    assert path.is_file(), f"Missing committed input: {path}"

print("Upload the paired-voice adapter files: adapter_config.json and adapter_model.safetensors")
uploaded = files.upload()
VOICE_ADAPTER.mkdir(parents=True, exist_ok=True)
for name, data in uploaded.items():
    if Path(name).name in {"adapter_config.json", "adapter_model.safetensors", "training_log.json"}:
        (VOICE_ADAPTER / Path(name).name).write_bytes(data)
assert (VOICE_ADAPTER / "adapter_config.json").is_file()
assert (VOICE_ADAPTER / "adapter_model.safetensors").is_file()

# Minimal adapter uploads do not contain tokenizer files.
AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(VOICE_ADAPTER)
print("Inputs ready")

In [ ]:
import gc, json, math
import torch.nn.functional as F
from sparse_autoencoders.run_sae import load_model, last_token_acts
from sparse_autoencoders.sae import SparseAutoencoder
from intervention.cot_utils import split_sentences
from evaluation.evaluate_ethics_morality import build_prompt

DEVICE = torch.device("cuda")
LAYER = 18
BATCH_SIZE = 16

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]

def readout_text(prompt, cot):
    return f"{prompt} {cot}\nFinal answer:"

def collect_pair_activations(adapter, bit1_texts, bit0_texts):
    tokenizer, model = load_model(str(adapter), DEVICE)
    h1 = last_token_acts(model, tokenizer, bit1_texts, LAYER, BATCH_SIZE)
    h0 = last_token_acts(model, tokenizer, bit0_texts, LAYER, BATCH_SIZE)
    del model, tokenizer
    gc.collect(); torch.cuda.empty_cache()
    return h1, h0

checkpoint = torch.load(SAE_PATH, map_location="cpu", weights_only=True)
assert int(checkpoint["layer"]) == LAYER and int(checkpoint["dict_size"]) == 7168
sae = SparseAutoencoder(896, 7168)
sae.load_state_dict(checkpoint["state_dict"])
sae.eval()

@torch.no_grad()
def encode(activations, batch_size=32):
    return torch.cat([sae.encode(activations[i:i + batch_size].float()) for i in range(0, len(activations), batch_size)])

print("Frozen SAE loaded:", LAYER, "896 → 7168")

In [ ]:
# S1 model: matched CoTs differ only in the first-sentence stance carrier.
s1_by_index = {int(row["index"]): row for row in read_jsonl(S1_RECORDS)}
s1_flips = sorted(read_jsonl(S1_FLIPS), key=lambda row: int(row["index"]))
assert len(s1_flips) == 100

s1_bit1_texts, s1_bit0_texts, s1_indices = [], [], []
for flip in s1_flips:
    index = int(flip["index"])
    record = s1_by_index[index]
    sentences = split_sentences(record["chain_of_thought"])
    assert sentences[0] == flip["original_first_sentence"]
    sentences[0] = flip["paraphrase"]
    flipped_cot = " ".join(sentences)
    original_text = readout_text(record["prompt"], record["chain_of_thought"])
    flipped_text = readout_text(record["prompt"], flipped_cot)
    assert int(flip["paraphrase_stance"]) == 1 - int(flip["original_stance"])
    bit1, bit0 = (
        (original_text, flipped_text)
        if int(flip["original_stance"]) == 1
        else (flipped_text, original_text)
    )
    s1_bit1_texts.append(bit1); s1_bit0_texts.append(bit0); s1_indices.append(index)

s1_h1, s1_h0 = collect_pair_activations(S1_ADAPTER, s1_bit1_texts, s1_bit0_texts)
assert s1_h1.shape == s1_h0.shape == (100, 896)
print("S1 activations:", s1_h1.shape)

In [ ]:
# Voice model: active is bit 1 and passive is bit 0 for the same scenario/content pair.
voice_groups = {}
for row in read_jsonl(VOICE_PAIRS):
    voice_groups.setdefault(int(row["pair_index"]), []).append(row)
assert len(voice_groups) == 100

voice_bit1_texts, voice_bit0_texts, voice_indices = [], [], []
for pair_index, pair in sorted(voice_groups.items()):
    assert len(pair) == 2
    by_voice = {row["voice"]: row for row in pair}
    assert set(by_voice) == {"active", "passive"}
    active, passive = by_voice["active"], by_voice["passive"]
    assert active["scenario"] == passive["scenario"]
    assert active["sentence_stances"] == passive["sentence_stances"]
    assert int(active["final_answer"]) == 1 and int(passive["final_answer"]) == 0
    voice_bit1_texts.append(readout_text(build_prompt(active), active["chain_of_thought"]))
    voice_bit0_texts.append(readout_text(build_prompt(passive), passive["chain_of_thought"]))
    voice_indices.append(pair_index)

voice_h1, voice_h0 = collect_pair_activations(VOICE_ADAPTER, voice_bit1_texts, voice_bit0_texts)
assert voice_h1.shape == voice_h0.shape == (100, 896)
print("Voice activations:", voice_h1.shape)

In [ ]:
# Encode with the same SAE, orient every difference as bit 1 minus bit 0, then decompose.
s1_z1, s1_z0 = encode(s1_h1), encode(s1_h0)
voice_z1, voice_z0 = encode(voice_h1), encode(voice_h0)

s1_delta_h = s1_h1 - s1_h0
voice_delta_h = voice_h1 - voice_h0
# SAE units have arbitrary scale; decoder norms convert feature changes to contribution scale.
decoder_norms = sae.decoder.weight.detach().norm(dim=0)
s1_delta_z = (s1_z1 - s1_z0) * decoder_norms
voice_delta_z = (voice_z1 - voice_z0) * decoder_norms

def decompose(s1_delta, voice_delta):
    task_matrix = torch.stack([
        F.normalize(s1_delta.mean(0), dim=0),
        F.normalize(voice_delta.mean(0), dim=0),
    ])
    pair_matrix = torch.cat([
        F.normalize(s1_delta, dim=1),
        F.normalize(voice_delta, dim=1),
    ])
    # Rows are sign-aligned and deliberately not mean-centered: their common mean is the signal.
    task_u, task_s, task_vh = torch.linalg.svd(task_matrix.cuda(), full_matrices=False)
    pair_u, pair_s, pair_vh = torch.linalg.svd(pair_matrix.cuda(), full_matrices=False)
    reference = task_matrix.mean(0).cuda()
    for u, vh in ((task_u, task_vh), (pair_u, pair_vh)):
        if torch.dot(vh[0], reference) < 0:
            vh[0].mul_(-1); u[:, 0].mul_(-1)
    return {
        "task_matrix": task_matrix.cpu(),
        "task_u": task_u.cpu(), "task_s": task_s.cpu(), "task_vh": task_vh.cpu(),
        "pair_matrix": pair_matrix.cpu(),
        "pair_u": pair_u.cpu(), "pair_s": pair_s.cpu(), "pair_vh": pair_vh.cpu(),
    }

residual_svd = decompose(s1_delta_h, voice_delta_h)
sae_svd = decompose(s1_delta_z, voice_delta_z)
print("Residual task SVD σ:", residual_svd["task_s"].tolist())
print("SAE task SVD σ:", sae_svd["task_s"].tolist())
print("SAE σ1/σ2:", float(sae_svd["task_s"][0] / sae_svd["task_s"][1]))

In [ ]:
# Save raw paired states, complete decompositions, shared directions, and feature rankings.
def cosine(a, b):
    return float(F.cosine_similarity(a, b, dim=0))

def svd_summary(result):
    singular = result["task_s"]
    return {
        "task_singular_values": singular.tolist(),
        "task_sigma1_over_sigma2": float(singular[0] / singular[1]),
        "pair_top10_singular_values": result["pair_s"][:10].tolist(),
    }

shared_sae = sae_svd["task_vh"][0]
values, indices = torch.topk(shared_sae.abs(), 50)
top_features = [
    {
        "rank": rank + 1,
        "feature": int(feature),
        "loading": float(shared_sae[feature]),
        "absolute_loading": float(value),
        "decoder_norm": float(decoder_norms[feature]),
    }
    for rank, (value, feature) in enumerate(zip(values, indices))
]

summary = {
    "base_model": BASE_MODEL,
    "s1_adapter": str(S1_ADAPTER),
    "voice_adapter": str(VOICE_ADAPTER),
    "sae": str(SAE_PATH),
    "layer": LAYER,
    "residual_dimension": 896,
    "sae_dimension": 7168,
    "s1_pairs": len(s1_indices),
    "voice_pairs": len(voice_indices),
    "orientation": "S1 stance 1 - 0; active(1) - passive(0)",
    "s1_voice_mean_cosine_residual": cosine(s1_delta_h.mean(0), voice_delta_h.mean(0)),
    "s1_voice_mean_cosine_sae_scaled": cosine(s1_delta_z.mean(0), voice_delta_z.mean(0)),
    "residual_svd": svd_summary(residual_svd),
    "sae_scaled_svd": svd_summary(sae_svd),
}

normalized_decoder = sae.decoder.weight.detach() / decoder_norms.clamp_min(1e-12)
shared_residual_from_sae = F.normalize(normalized_decoder @ shared_sae, dim=0)

torch.save({
    "s1_indices": s1_indices, "voice_pair_indices": voice_indices,
    "s1_h1": s1_h1, "s1_h0": s1_h0, "s1_delta_h": s1_delta_h,
    "voice_h1": voice_h1, "voice_h0": voice_h0, "voice_delta_h": voice_delta_h,
    "s1_z1": s1_z1, "s1_z0": s1_z0, "s1_delta_z_scaled": s1_delta_z,
    "voice_z1": voice_z1, "voice_z0": voice_z0, "voice_delta_z_scaled": voice_delta_z,
    "decoder_norms": decoder_norms,
}, RESULTS / "paired_activations.pt")
torch.save({
    "residual": residual_svd,
    "sae_scaled": sae_svd,
    "shared_residual_direction": residual_svd["task_vh"][0],
    "shared_sae_direction": shared_sae,
    "shared_residual_from_sae": shared_residual_from_sae,
}, RESULTS / "svd_results.pt")
(RESULTS / "summary.json").write_text(json.dumps(summary, indent=2))
(RESULTS / "top_shared_sae_features.json").write_text(json.dumps(top_features, indent=2))
print(json.dumps(summary, indent=2))
print("Top shared SAE features:", [row["feature"] for row in top_features[:10]])

In [ ]:
import shutil

for name in ("paired_activations.pt", "svd_results.pt", "summary.json", "top_shared_sae_features.json"):
    path = RESULTS / name
    assert path.is_file() and path.stat().st_size > 0, path

archive = shutil.make_archive("/content/sae_svd_05b_l18", "zip", root_dir=RESULTS)
files.download(archive)
print("Downloaded", archive)